In [1]:
import numpy as np
from shape_model.gaussian_ellipsoid import gaussian_ellipsoid_shape_model
from analytical_scattering_theories.homogeneous_sphere import mie_compute_q_and_s
from bl_dda.scatterer import Target, IncidentField, DiscreteDipoles



In [2]:
# Define the shape parameters of Gaussian Random Ellipsoid (a >= b >= c)
#### length_unit= um ###
r_v_base= 0.2 # allowed range = [0.1, 0.5], volume equivalent radius of base ellipsoid
bc_ratio= 1 # allowed range = [1.0, 7.0], b/c ratio of base ellipsoid
ab_ratio= 1 # allowed range = [1.0, 2.0], a/b ratio of base ellipsoid
beta= 0.0 # standard deviation of GE surface deformation, allowed range= [0, 0.3]

# Define wavelength and refractive index

# vacuum wavelength [um]
wl_0= 0.834  

# medium refractive index
m_m=1.329 
#m_m=1.0

# particle's complex refractive index
m_p_x= 1.5+0.01j  # refractive index along x axis in particle coordinate system
m_p_y= 1.5+0.01j  # refractive index along y axis in particle coordinate system
m_p_z= 1.5+0.01j  # refractive index along z axis in particle coordinate system
m_p_xyz= np.array([m_p_x, m_p_y, m_p_z], dtype=np.complex64)

# axis-averaged particle's complex refractive index
m_p_avg= (m_p_x+m_p_y+m_p_z)/3

In [3]:
# generate the GRE particle volume in 3D lattice
gre_shape_model= gaussian_ellipsoid_shape_model(r_v_base, bc_ratio, ab_ratio, beta, wl_0, m_p_xyz)

rng= np.random.default_rng(1234)
r_points_on_GRE_surf, xyz_meshes_GRE_surf = gre_shape_model.compute_r_points_on_GRE(rng)
lattice_domain, lattice_n, lattice_grid_points= gre_shape_model.create_cuboid_lattice_that_encloses_GRE_shape(r_points_on_GRE_surf)
dist_from_GRE= gre_shape_model.find_nearest_distance_from_the_GRE_surf(lattice_grid_points, r_points_on_GRE_surf)
lattice_grid_points_is_in_target= gre_shape_model.extract_lattice_address_in_GRE_volume(gre_shape_model.lattice_lf, gre_shape_model.distance_factor, lattice_n, dist_from_GRE)

In [4]:
target= Target(gre_shape_model.name, lattice_n, gre_shape_model.lattice_lf, lattice_grid_points, lattice_grid_points_is_in_target, m_p_xyz, r_v_base)
lattice_n
# np.prod(lattice_n)= 269705
lattice_n, np.prod(lattice_n)

(array([15, 15, 15], dtype=int32), np.int64(3375))

In [5]:
# Define incident fields using deterministic Euler angle grid
# Uniform on SO(3): alpha equally spaced, cos(beta) equally spaced, gamma equally spaced

N_alpha_ori = 3
N_beta_ori  = 4
N_gamma_ori = 3
number_of_orientations = N_alpha_ori * N_beta_ori * N_gamma_ori

alpha = np.linspace(0, 2*np.pi, N_alpha_ori, endpoint=False)
cos_beta = np.linspace(1 - 1/N_beta_ori, -1 + 1/N_beta_ori, N_beta_ori)
beta_arr = np.arccos(cos_beta)
gamma = np.linspace(0, 2*np.pi, N_gamma_ori, endpoint=False)

aa, bb, gg = np.meshgrid(alpha, beta_arr, gamma, indexing='ij')
euler_angles_radian = np.column_stack([aa.ravel(), bb.ravel(), gg.ravel()])

incident_field = IncidentField(wl_0, m_m, euler_angles_radian)

print(f"Orientation grid: N_alpha={N_alpha_ori}, N_beta={N_beta_ori}, N_gamma={N_gamma_ori}")
print(f"Total orientations: {number_of_orientations}")
print(f"beta values [deg]: {np.degrees(beta_arr)}")

Orientation grid: N_alpha=3, N_beta=4, N_gamma=3
Total orientations: 36
beta values [deg]: [ 41.40962211  75.52248781 104.47751219 138.59037789]


In [6]:
discrete_dipoles= DiscreteDipoles(target, incident_field)

Number of dipoles per wavelength in the particle volume: dpl= 17.245360770293434


In [7]:
discrete_dipoles.set_interaction_matrix()

In [8]:
discrete_dipoles.solve_matrix_equation()

Starting block-GMRES iterative solver...


block-GMRES converged! (iter_fin=1, err_fin=0.0065, solver time=0.3s)


In [9]:
# Compute Mie solution for the volume-equivalent sphere

r_p= discrete_dipoles.ve_radius # volume-equivalent radius of the DDA target
Q_sca_mie, Q_abs_mie, Q_ext_mie, S_fw_PCAS_mie, S_bk_OCBS_mie = mie_compute_q_and_s(wl_0,m_m,r_p,m_p_avg,nang=3)
print(Q_sca_mie, Q_abs_mie, Q_ext_mie)
print(S_fw_PCAS_mie, S_bk_OCBS_mie)

0.09515763335751952 0.04425046431495959 0.1394080976724791
(0.070486106+0.013958116j) (0.0010374496+0.0026707908j)


In [10]:
# Compare the absorption efficiency Q_abs between DDA solutions (for each orientation) and Mie solution
# Tolerance: |Q_abs/Q_abs_mie - 1| < 0.01 for every orientation

C_abs= discrete_dipoles.compute_C_abs()
Q_abs= C_abs/(np.pi*discrete_dipoles.ve_radius**2)
print(np.abs(Q_abs/Q_abs_mie - 1))

[0.00267862 0.0043587  0.00323929 0.00048977 0.00351055 0.00232134
 0.00111706 0.00297522 0.00254166 0.00389091 0.00311873 0.00359859
 0.00267862 0.0043587  0.00323929 0.00048977 0.00351055 0.00232134
 0.00111706 0.00297522 0.00254166 0.00389091 0.00311873 0.00359859
 0.00267862 0.0043587  0.00323929 0.00048977 0.00351055 0.00232134
 0.00111706 0.00297522 0.00254166 0.00389091 0.00311873 0.00359859]


In [11]:
# Compare the absorption efficiency Q_ext between DDA solutions (for each orientation) and Mie solution
# Tolerance: |Q_ext/Q_ext_mie - 1| < 0.01 for every orientation

C_ext= discrete_dipoles.compute_C_ext()
Q_ext= C_ext/(np.pi*discrete_dipoles.ve_radius**2)
print(np.abs(Q_ext/Q_ext_mie - 1))

[0.00212204 0.00295311 0.00251312 0.00047439 0.00254816 0.0019841
 0.00068822 0.00233173 0.00209614 0.00248406 0.00242609 0.00269044
 0.00212204 0.00295311 0.00251312 0.00047439 0.00254816 0.0019841
 0.00068822 0.00233173 0.00209614 0.00248406 0.00242609 0.00269044
 0.00212204 0.00295311 0.00251312 0.00047439 0.00254816 0.0019841
 0.00068822 0.00233173 0.00209614 0.00248406 0.00242609 0.00269044]


In [12]:
# Compare the polarized forward complex scattering amplitude between DDA solutions (for each orientation) and Mie solution
# Tolerance for theta-polarization (s-pol in PCAS): |S_fw_theta_dda.real/S_fw_mie.real - 1| < 0.01 and  |S_fw_theta_dda.imag/S_fw_mie.imag - 1| < 0.01 for every orientation
# Tolerance for phi-polarization (p-pol in PCAS): |S_fw_phi_dda.real/S_fw_mie.real - 1| < 0.01 and  |S_fw_phi_dda.imag/S_fw_mie.imag - 1| < 0.01 for every orientation

S_fw_PCAS_theta_P, S_fw_PCAS_phi_P= discrete_dipoles.compute_PCAS_observable_S_fw()
print(np.abs(S_fw_PCAS_theta_P.real/S_fw_PCAS_mie.real - 1), np.abs(S_fw_PCAS_theta_P.imag/S_fw_PCAS_mie.imag - 1))
print(np.abs(S_fw_PCAS_phi_P.real/S_fw_PCAS_mie.real - 1), np.abs(S_fw_PCAS_phi_P.imag/S_fw_PCAS_mie.imag - 1))

[2.43667828e-03 2.24958619e-03 2.03372258e-03 2.46755268e-04
 3.46978985e-04 3.82522188e-04 5.18780214e-04 8.24053743e-05
 5.68611715e-04 2.96511532e-03 1.55501220e-03 2.49200909e-03
 4.83809701e-04 1.71209886e-03 1.58362039e-04 1.82940660e-04
 2.22727840e-03 2.38142319e-04 2.08301569e-04 1.71437115e-03
 6.93556989e-04 4.55790952e-04 1.05981886e-03 6.51914213e-04
 4.56568920e-04 2.08418274e-03 1.97632992e-03 2.68133717e-04
 2.16786014e-03 2.30055206e-03 9.00664421e-04 1.98457491e-03
 1.99397089e-03 1.78148373e-03 1.44202450e-03 1.66741824e-03] [0.00220148 0.00186831 0.00278719 0.00084071 0.0027214  0.00402889
 0.00254498 0.00154396 0.00169523 0.00138101 0.00131177 0.00027027
 0.00365099 0.00247089 0.0023458  0.00053674 0.00785698 0.00757617
 0.00180162 0.00787756 0.00625182 0.00096683 0.00209668 0.00028634
 0.00781568 0.00452018 0.00798078 0.00172719 0.00293388 0.00240505
 0.00280805 0.00242628 0.00173188 0.00980006 0.00386985 0.00805528]
[1.85307013e-04 1.78099234e-03 7.45220448e-04 2

In [13]:
S_bk_OCBS_P= discrete_dipoles.compute_OCBS_observable_S_bk()

In [14]:
S_bk_OCBS_P, S_bk_OCBS_mie


(array([0.00141458+0.00274476j, 0.00170766+0.00358323j,
        0.00164769+0.00268315j, 0.00012153+0.00306956j,
        0.00139151+0.00335318j, 0.00108935+0.0031376j ,
        0.00037719+0.00276797j, 0.00143839+0.00244267j,
        0.00124264+0.00306384j, 0.00165981+0.00302212j,
        0.00147748+0.00300891j, 0.0016142 +0.00335998j,
        0.00141458+0.00274476j, 0.00170766+0.00358323j,
        0.00164769+0.00268315j, 0.00012153+0.00306956j,
        0.00139151+0.00335318j, 0.00108935+0.0031376j ,
        0.00037719+0.00276797j, 0.00143839+0.00244267j,
        0.00124264+0.00306384j, 0.00165981+0.00302212j,
        0.00147748+0.00300891j, 0.0016142 +0.00335998j,
        0.00141458+0.00274476j, 0.00170766+0.00358323j,
        0.00164769+0.00268315j, 0.00012153+0.00306956j,
        0.00139151+0.00335318j, 0.00108935+0.0031376j ,
        0.00037719+0.00276797j, 0.00143839+0.00244267j,
        0.00124264+0.00306384j, 0.00165981+0.00302212j,
        0.00147748+0.00300891j, 0.0016142 +0.003